In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
Implement decode-phase multi-head attention where the key and value caches are stored as
<code>int8</code> with per-token scale factors. This memory layout halves KV-cache bandwidth
versus <code>float32</code> and is used in production LLM serving systems such as TensorRT-LLM
and vLLM. Given a query tensor <code>Q</code> for a single new token, <code>int8</code> key cache
<code>K_int8</code>, <code>int8</code> value cache <code>V_int8</code>, and per-token scales
<code>k_scale</code> and <code>v_scale</code>, dequantize the caches and compute scaled
dot-product attention to produce <code>output</code>. All non-integer tensors use
<code>float32</code>.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Implement the function <code>solve(Q, K_int8, V_int8, k_scale, v_scale, output, num_heads, seq_len, head_dim)</code>.</li>
  <li>Do not change the function signature or use external libraries beyond the standard GPU frameworks.</li>
  <li>Write the result into the provided <code>output</code> buffer.</li>
  <li>Dequantize using per-token scales: <code>K_float[h, s, d] = K_int8[h, s, d] &times; k_scale[h, s]</code> (and analogously for V).</li>
  <li>Use scaled dot-product attention with scale factor <code>1 / sqrt(head_dim)</code> and a softmax over the sequence dimension.</li>
</ul>

<h2>Example</h2>
<p>
  With <code>num_heads</code> = 1, <code>seq_len</code> = 3, <code>head_dim</code> = 4:
</p>
<p>
  <strong>Input:</strong><br>
  $Q$ (1&times;4):
  $$
  \begin{bmatrix} 1 & 1 & 1 & 1 \end{bmatrix}
  $$
  $K\_int8$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 10 & 0 & 0 & 0 \\ 0 & 10 & 0 & 0 \\ 0 & 0 & 10 & 0 \end{bmatrix}
  $$
  $k\_scale$ (1&times;3): $\begin{bmatrix} 0.1 & 0.1 & 0.1 \end{bmatrix}$
  &nbsp;&rArr;&nbsp;
  $K\_float$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \end{bmatrix}
  $$
  $V\_int8$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 10 & 20 & 30 & 40 \\ 50 & 60 & 70 & 80 \\ 90 & 100 & 110 & 120 \end{bmatrix}
  $$
  $v\_scale$ (1&times;3): $\begin{bmatrix} 0.1 & 0.1 & 0.1 \end{bmatrix}$
  &nbsp;&rArr;&nbsp;
  $V\_float$ (1&times;3&times;4):
  $$
  \begin{bmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \end{bmatrix}
  $$
</p>
<p>
  Scores = $Q \cdot K\_float^T / \sqrt{4}$ = $\begin{bmatrix} 0.5 & 0.5 & 0.5 \end{bmatrix}$,
  so <em>softmax</em> weights = $\begin{bmatrix} 1/3 & 1/3 & 1/3 \end{bmatrix}$.
</p>
<p>
  <strong>Output</strong> (1&times;4):
  $$
  \begin{bmatrix} 5.00 & 6.00 & 7.00 & 8.00 \end{bmatrix}
  $$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>num_heads</code> &le; 64</li>
  <li>1 &le; <code>seq_len</code> &le; 32,768</li>
  <li>8 &le; <code>head_dim</code> &le; 256; <code>head_dim</code> is a multiple of 8</li>
  <li><code>K_int8</code> and <code>V_int8</code> values are in $[-128, 127]$</li>
  <li>All scale values are positive <code>float32</code></li>
  <li>Performance is measured with <code>num_heads</code> = 32, <code>seq_len</code> = 8,192, <code>head_dim</code> = 128</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, K_int8, V_int8, k_scale, v_scale, output are device pointers
extern "C" void solve(const float* Q, const int8_t* K_int8, const int8_t* V_int8,
                      const float* k_scale, const float* v_scale, float* output, int num_heads,
                      int seq_len, int head_dim) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, K_int8, V_int8, k_scale, v_scale, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    K_int8: cute.Tensor,
    V_int8: cute.Tensor,
    k_scale: cute.Tensor,
    v_scale: cute.Tensor,
    output: cute.Tensor,
    num_heads: cute.Int32,
    seq_len: cute.Int32,
    head_dim: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, K_int8, V_int8, k_scale, v_scale are tensors on GPU
@jax.jit
def solve(
    Q: jax.Array,
    K_int8: jax.Array,
    V_int8: jax.Array,
    k_scale: jax.Array,
    v_scale: jax.Array,
    num_heads: int,
    seq_len: int,
    head_dim: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer


# Q, K_int8, V_int8, k_scale, v_scale, output are device pointers
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    K_int8: UnsafePointer[Int8, MutExternalOrigin],
    V_int8: UnsafePointer[Int8, MutExternalOrigin],
    k_scale: UnsafePointer[Float32, MutExternalOrigin],
    v_scale: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    num_heads: Int32,
    seq_len: Int32,
    head_dim: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, K_int8, V_int8, k_scale, v_scale, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K_int8: torch.Tensor,
    V_int8: torch.Tensor,
    k_scale: torch.Tensor,
    v_scale: torch.Tensor,
    output: torch.Tensor,
    num_heads: int,
    seq_len: int,
    head_dim: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, K_int8, V_int8, k_scale, v_scale, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K_int8: torch.Tensor,
    V_int8: torch.Tensor,
    k_scale: torch.Tensor,
    v_scale: torch.Tensor,
    output: torch.Tensor,
    num_heads: int,
    seq_len: int,
    head_dim: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
import math
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="INT8 KV-Cache Attention",
            atol=1e-03,
            rtol=1e-03,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        Q: torch.Tensor,
        K_int8: torch.Tensor,
        V_int8: torch.Tensor,
        k_scale: torch.Tensor,
        v_scale: torch.Tensor,
        output: torch.Tensor,
        num_heads: int,
        seq_len: int,
        head_dim: int,
    ):
        assert Q.shape == (num_heads, head_dim)
        assert K_int8.shape == (num_heads, seq_len, head_dim)
        assert V_int8.shape == (num_heads, seq_len, head_dim)
        assert k_scale.shape == (num_heads, seq_len)
        assert v_scale.shape == (num_heads, seq_len)
        assert output.shape == (num_heads, head_dim)
        assert Q.dtype == torch.float32
        assert K_int8.dtype == torch.int8
        assert V_int8.dtype == torch.int8
        assert k_scale.dtype == torch.float32
        assert v_scale.dtype == torch.float32
        assert output.dtype == torch.float32
        assert Q.device.type == "cuda"
        assert K_int8.device.type == "cuda"
        assert V_int8.device.type == "cuda"
        assert k_scale.device.type == "cuda"
        assert v_scale.device.type == "cuda"
        assert output.device.type == "cuda"

        # Dequantize: K_float[h, s, d] = K_int8[h, s, d] * k_scale[h, s]
        K_float = K_int8.float() * k_scale.unsqueeze(-1)  # [num_heads, seq_len, head_dim]
        V_float = V_int8.float() * v_scale.unsqueeze(-1)  # [num_heads, seq_len, head_dim]

        # Scaled dot-product attention: Q [num_heads, head_dim] attends to all seq_len positions
        scale = 1.0 / math.sqrt(head_dim)
        # scores: [num_heads, 1, seq_len]
        scores = torch.bmm(Q.unsqueeze(1), K_float.transpose(1, 2)) * scale
        weights = torch.softmax(scores, dim=-1)  # [num_heads, 1, seq_len]

        # Weighted sum of V: [num_heads, 1, seq_len] @ [num_heads, seq_len, head_dim]
        out = torch.bmm(weights, V_float)  # [num_heads, 1, head_dim]
        output.copy_(out.squeeze(1))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "Q": (ctypes.POINTER(ctypes.c_float), "in"),
            "K_int8": (ctypes.POINTER(ctypes.c_int8), "in"),
            "V_int8": (ctypes.POINTER(ctypes.c_int8), "in"),
            "k_scale": (ctypes.POINTER(ctypes.c_float), "in"),
            "v_scale": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "num_heads": (ctypes.c_int, "in"),
            "seq_len": (ctypes.c_int, "in"),
            "head_dim": (ctypes.c_int, "in"),
        }

    def _make_test_case(self, num_heads, seq_len, head_dim, zero_q=False, seed=None):
        device = "cuda"
        if seed is not None:
            torch.manual_seed(seed)
        if zero_q:
            Q = torch.zeros(num_heads, head_dim, dtype=torch.float32, device=device)
        else:
            Q = torch.randn(num_heads, head_dim, dtype=torch.float32, device=device)
        K_int8 = torch.randint(
            -128, 128, (num_heads, seq_len, head_dim), dtype=torch.int8, device=device
        )
        V_int8 = torch.randint(
            -128, 128, (num_heads, seq_len, head_dim), dtype=torch.int8, device=device
        )
        k_scale = torch.rand(num_heads, seq_len, dtype=torch.float32, device=device) * 0.1 + 0.01
        v_scale = torch.rand(num_heads, seq_len, dtype=torch.float32, device=device) * 0.1 + 0.01
        output = torch.empty(num_heads, head_dim, dtype=torch.float32, device=device)
        return {
            "Q": Q,
            "K_int8": K_int8,
            "V_int8": V_int8,
            "k_scale": k_scale,
            "v_scale": v_scale,
            "output": output,
            "num_heads": num_heads,
            "seq_len": seq_len,
            "head_dim": head_dim,
        }

    def generate_example_test(self) -> Dict[str, Any]:
        device = "cuda"
        num_heads, seq_len, head_dim = 1, 3, 4
        Q = torch.tensor([[1.0, 1.0, 1.0, 1.0]], dtype=torch.float32, device=device)
        K_int8 = torch.tensor(
            [[[10, 0, 0, 0], [0, 10, 0, 0], [0, 0, 10, 0]]], dtype=torch.int8, device=device
        )
        V_int8 = torch.tensor(
            [[[10, 20, 30, 40], [50, 60, 70, 80], [90, 100, 110, 120]]],
            dtype=torch.int8,
            device=device,
        )
        k_scale = torch.tensor([[0.1, 0.1, 0.1]], dtype=torch.float32, device=device)
        v_scale = torch.tensor([[0.1, 0.1, 0.1]], dtype=torch.float32, device=device)
        output = torch.empty(num_heads, head_dim, dtype=torch.float32, device=device)
        return {
            "Q": Q,
            "K_int8": K_int8,
            "V_int8": V_int8,
            "k_scale": k_scale,
            "v_scale": v_scale,
            "output": output,
            "num_heads": num_heads,
            "seq_len": seq_len,
            "head_dim": head_dim,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        tests = []
        # Edge: single key in cache
        tests.append(self._make_test_case(1, 1, 8, seed=0))
        # Edge: two keys
        tests.append(self._make_test_case(1, 2, 8, seed=1))
        # Edge: four keys, two heads
        tests.append(self._make_test_case(2, 4, 8, seed=2))
        # Zero query (uniform softmax weights)
        tests.append(self._make_test_case(1, 8, 16, zero_q=True, seed=3))
        # Power-of-2 seq_len
        tests.append(self._make_test_case(4, 16, 64, seed=4))
        tests.append(self._make_test_case(8, 64, 64, seed=5))
        # Non-power-of-2
        tests.append(self._make_test_case(2, 30, 64, seed=6))
        tests.append(self._make_test_case(4, 100, 64, seed=7))
        # Realistic sizes
        tests.append(self._make_test_case(16, 512, 64, seed=8))
        tests.append(self._make_test_case(32, 256, 128, seed=9))
        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        return self._make_test_case(32, 8192, 128, seed=42)


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
